# TinyStories CPU benchmark

Use a **Colab CPU runtime** with two available CPU threads and a recent PyTorch that supports `torch.nn.RMSNorm`. The original runs used PyTorch `2.14.1+cpu`.

The default `SELECTED_SEEDS = (1776,)` runs one matched pair, about 60 minutes of measured training plus setup and evaluation. Set it to `(1776, 2027, 31415)` to run all six models. Run the cells in order. Both models in a pair must finish on the same runtime. If the runtime is replaced, start that pair from scratch.

The code uses a dedicated workspace for this reproduction. Starting it on a fresh VM avoids reusing checkpoints. Each pair prints `PAIR RESULT`. The two checkpoints measure validation bits per byte at 10,567,680 processed bytes and after the first complete step at or beyond 1,800 training seconds. A slower CPU may miss the equal-byte checkpoint.


In [ ]:
WORKER_SOURCE = r"""


import hashlib
import json
import math
import os
import sys
from pathlib import Path

BATCH_SIZE = 8
CONTEXT = 128
VOCAB_SIZE = 256
HEAD_WIDTH = 16
TOKENS_PER_STEP = BATCH_SIZE * CONTEXT
MODELS = {
    "shallow_wide": (3, 96, 6, 357_024),
    "deep_narrow": (7, 64, 4, 361_408),
}


def setup_torch():
    global torch, nn, F
    import torch
    from torch import nn
    from torch.nn import functional as F

    visible_cpus = len(os.sched_getaffinity(0))
    requested_threads = min(2, visible_cpus)
    try:
        torch.set_num_threads(requested_threads)
        if torch.get_num_threads() != requested_threads:
            torch.set_num_threads(1)
    except (RuntimeError, ValueError) as error:
        if requested_threads == 1:
            raise
        print(f"Two-thread setup failed ({error}). Trying one thread.", flush=True)
        torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
    if torch.get_num_threads() not in (1, 2) or torch.get_num_interop_threads() != 1:
        raise RuntimeError("Could not establish the requested CPU thread configuration.")


def make_models(seed):
    def rotate_half(x):
        left, right = x.chunk(2, dim=-1)
        return torch.cat((-right, left), dim=-1)

    class Block(nn.Module):
        def __init__(self, width, heads):
            super().__init__()
            self.heads = heads
            self.attention_norm = nn.RMSNorm(width, eps=1e-5)
            self.qkv = nn.Linear(width, 3 * width, bias=False)
            self.attention_output = nn.Linear(width, width, bias=False)
            self.ffn_norm = nn.RMSNorm(width, eps=1e-5)
            self.expand = nn.Linear(width, 4 * width, bias=False)
            self.contract = nn.Linear(4 * width, width, bias=False)

        def forward(self, x, cos, sin):
            batch, length, width = x.shape
            qkv = self.qkv(self.attention_norm(x)).reshape(
                batch, length, 3, self.heads, HEAD_WIDTH
            )
            q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            attended = F.scaled_dot_product_attention(
                q, k, v, dropout_p=0.0, is_causal=True
            )
            attended = attended.transpose(1, 2).reshape(batch, length, width)
            x = x + self.attention_output(attended)
            return x + self.contract(F.gelu(self.expand(self.ffn_norm(x)), approximate="tanh"))

    class ByteTransformer(nn.Module):
        def __init__(self, layers, width, heads):
            super().__init__()
            assert width == heads * HEAD_WIDTH
            self.embedding = nn.Embedding(VOCAB_SIZE, width)
            self.blocks = nn.ModuleList(Block(width, heads) for _ in range(layers))
            self.final_norm = nn.RMSNorm(width, eps=1e-5)
            inverse_frequency = 10_000 ** (
                -torch.arange(0, HEAD_WIDTH, 2, dtype=torch.float32) / HEAD_WIDTH
            )
            angles = torch.outer(torch.arange(CONTEXT, dtype=torch.float32), inverse_frequency)
            angles = torch.cat((angles, angles), dim=-1)[None, None, :, :]
            self.register_buffer("cos", angles.cos(), persistent=False)
            self.register_buffer("sin", angles.sin(), persistent=False)
            for module in self.modules():
                if isinstance(module, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(module.weight, std=0.02)

        def forward(self, tokens):
            x = self.embedding(tokens)
            for block in self.blocks:
                x = block(x, self.cos, self.sin)
            # The output projection uses the input embedding parameter directly.
            return F.linear(self.final_norm(x), self.embedding.weight)

    models = {}
    for name, (layers, width, heads, expected) in MODELS.items():
        torch.manual_seed(seed)
        model = ByteTransformer(layers, width, heads).train()
        actual = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if actual != expected:
            raise ValueError(f"{name}: expected {expected:,} parameters, got {actual:,}")
        models[name] = model
        print(f"{name}: verified {actual:,} trainable parameters", flush=True)
    return models


def training_step(model, optimizer, tokens, labels):
    optimizer.zero_grad(set_to_none=True)
    logits = model(tokens)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), labels)
    loss.backward()
    optimizer.step()
    return loss.detach()



import requests
from time import perf_counter

REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
SOURCE = f"https://huggingface.co/datasets/roneneldan/TinyStories/resolve/{REVISION}"
SEEDS = (1776, 2027, 31415)
FULL_STEPS = 10320
TRAIN_BYTES = 64 * 1024 * 1024
EVAL_BYTES = 128 * 1024
ADAMW = dict(lr=3e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1,
             foreach=False, fused=False)


def save_json(path, data):
    pending = path.with_suffix(".tmp")
    pending.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")
    pending.replace(path)


def prefix(cache, name, length):
    target = cache / name
    if not target.exists():
        print(f"Reading {length:,} bytes from {name} at revision {REVISION}", flush=True)
        with requests.get(f"{SOURCE}/{name}", stream=True, timeout=(15, 45),
                          headers={"Accept-Encoding": "identity"}) as response:
            response.raise_for_status()
            data = response.raw.read(length, decode_content=True)
        if len(data) != length:
            raise ValueError(f"{name} returned {len(data):,} bytes, expected {length:,}")
        pending = target.with_suffix(".tmp")
        pending.write_bytes(data)
        pending.replace(target)
    data = target.read_bytes()
    if len(data) != length:
        raise ValueError(f"Incomplete cached data in {target}")
    return data


def as_batches(data):
    stride = BATCH_SIZE * (CONTEXT + 1)
    size = len(data) // stride * stride
    return torch.frombuffer(bytearray(data[:size]), dtype=torch.uint8).view(
        -1, BATCH_SIZE, CONTEXT + 1
    )


def xy(batch):
    return batch[:, :-1].long(), batch[:, 1:].long().reshape(-1)


def evaluate(model, batches):
    model.eval()
    loss = 0.0
    with torch.inference_mode():
        for batch in batches:
            x, y = xy(batch)
            logits = model(x).reshape(-1, VOCAB_SIZE)
            loss += float(F.cross_entropy(logits, y, reduction="sum"))
    model.train()
    score = loss / (len(batches) * TOKENS_PER_STEP * math.log(2))
    if not math.isfinite(score):
        raise ValueError("Nonfinite bits per byte")
    return score


def checkpoint(path, model, optimizer, step, elapsed, scores):
    temp = path.with_suffix(".tmp")
    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "step": step, "training_seconds": elapsed, "scores": scores,
    }, temp)
    temp.replace(path)


def train_one(out, name, model, seed, train, order, validation, target_steps, seconds):
    optimizer = torch.optim.AdamW(model.parameters(), **ADAMW)
    elapsed = 0.0
    step = 0
    scores = {}
    run = out / f"seed_{seed}_{name}"
    run.mkdir(exist_ok=True)
    latest = run / "latest.pt"
    if latest.exists():
        state = torch.load(latest, map_location="cpu", weights_only=True)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        step, elapsed, scores = state["step"], state["training_seconds"], state["scores"]
    print(f"TRAINING {name} | seed {seed} | {seconds} s budget | starting at step {step}", flush=True)
    next_save = (int(elapsed) // 60 + 1) * 60

    def evaluate_checkpoint(label):
        if label in scores:
            return
        path = run / f"{label}.pt"
        checkpoint(path, model, optimizer, step, elapsed, scores)
        restored = torch.load(path, map_location="cpu", weights_only=True)
        model.load_state_dict(restored["model"])
        optimizer.load_state_dict(restored["optimizer"])
        assert restored["step"] == step and restored["training_seconds"] == elapsed
        row = {"seed": seed, "model": name, "checkpoint": label, "steps": step,
               "bytes": step * TOKENS_PER_STEP, "training_seconds": elapsed,
               "bits_per_byte": evaluate(model, validation)}
        scores[label] = row
        checkpoint(latest, model, optimizer, step, elapsed, scores)
        save_json(run / f"{label}.json", row)
        print("RESULT " + json.dumps(row), flush=True)

    while elapsed < seconds:
        if step >= len(order):
            raise RuntimeError("Exhausted the fixed training prefix")
        start = perf_counter()
        x, y = xy(train[order[step]])
        loss = training_step(model, optimizer, x, y)
        elapsed += perf_counter() - start
        step += 1
        if step == target_steps or elapsed >= next_save or elapsed >= seconds:
            if not math.isfinite(float(loss)):
                raise ValueError(f"Nonfinite training loss at step {step}")
        if step == target_steps:
            evaluate_checkpoint("equal_bytes")
        if elapsed >= next_save:
            checkpoint(latest, model, optimizer, step, elapsed, scores)
            print(f"PROGRESS {name} | seed {seed} | {elapsed:.1f} s | {step * TOKENS_PER_STEP:,} bytes",
                  flush=True)
            next_save += 60
    evaluate_checkpoint("equal_time")
    if "equal_bytes" not in scores:
        print(f"MISSING equal-bytes checkpoint for {name}, seed {seed}", flush=True)
    return list(scores.values())



TRAIN_SHA256 = "70163c7345ec1da54cf495d9e554227eef6a622e02de79e5bacf6d8559cd292f"
VALID_SHA256 = "4a79ede43ee5a4931fb3c5ae06f55406aecc8f6737be6ce3ce606dd84f5e86bd"


def load_study_data(workspace):
    cache = workspace / "data"
    cache.mkdir(exist_ok=True)
    source = prefix(cache, "TinyStories-train.txt", TRAIN_BYTES + 2 * EVAL_BYTES)
    delimiter = b"<|endoftext|>"
    boundary = source.index(delimiter, EVAL_BYTES) + len(delimiter)
    train = source[boundary:boundary + TRAIN_BYTES]
    validation = prefix(cache, "TinyStories-valid.txt", EVAL_BYTES)
    if hashlib.sha256(train).hexdigest() != TRAIN_SHA256:
        raise ValueError("Training bytes differ from the prior run.")
    if hashlib.sha256(validation).hexdigest() != VALID_SHA256:
        raise ValueError("Validation bytes differ from the prior run.")
    return as_batches(train), as_batches(validation)


def cpu_model():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.startswith(("model name", "Hardware")):
            return line.split(":", 1)[1].strip()
    return os.uname().machine


def run_model(workspace, seed, name):
    train, validation = load_study_data(workspace)
    runtime = (workspace / "runtime_id.txt").read_text().strip()
    metadata = {
        "runtime_id": runtime, "seed": seed, "model": name, "cpu": cpu_model(),
        "threads": torch.get_num_threads(), "torch": str(torch.__version__),
        "revision": REVISION, "train_sha256": TRAIN_SHA256,
        "eval_sha256": VALID_SHA256, "eval_bytes": len(validation) * TOKENS_PER_STEP,
        "target_bytes": FULL_STEPS * TOKENS_PER_STEP, "time_budget_s": 1800,
    }
    if metadata["threads"] != 2:
        raise RuntimeError("This comparison requires two available PyTorch CPU threads.")
    print("RUN " + json.dumps(metadata), flush=True)
    models = make_models(seed)
    order = torch.randperm(len(train), generator=torch.Generator().manual_seed(seed)).tolist()
    out = workspace / "results"
    out.mkdir(exist_ok=True)
    scores = train_one(out, name, models[name], seed, train, order, validation, FULL_STEPS, 1800)
    record = {**metadata, "measurements": scores}
    save_json(out / f"seed_{seed}_{name}.json", record)
    print("MODEL COMPLETE " + json.dumps(record), flush=True)


if __name__ == "__main__":
    setup_torch()
    torch.set_default_dtype(torch.float32)
    workspace = Path(sys.argv[1])
    seed, name = int(sys.argv[2]), sys.argv[3]
    if seed not in SEEDS or name not in MODELS:
        raise ValueError("Unknown seed or model.")
    run_model(workspace, seed, name)
"""

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from threading import Thread
from uuid import uuid4

SELECTED_SEEDS = (1776,)

workspace = Path('/content/tinystories_cpu_benchmark')
workspace.mkdir(exist_ok=True)
runtime_file = workspace / 'runtime_id.txt'
if not runtime_file.exists():
    runtime_file.write_text(uuid4().hex)
worker_file = workspace / 'worker.py'
worker_file.write_text(WORKER_SOURCE)

def train_model(seed, name, predecessor=None):
    if seed not in SELECTED_SEEDS:
        return
    if predecessor is not None:
        previous = workspace / 'results' / f'seed_{seed}_{predecessor}.json'
        if not previous.exists():
            raise RuntimeError(f'Run both models for seed {seed} on this same runtime.')
    env = dict(os.environ, OMP_NUM_THREADS='2', MKL_NUM_THREADS='2',
               OPENBLAS_NUM_THREADS='2')
    command = [sys.executable, '-u', str(worker_file), str(workspace), str(seed), name]
    with subprocess.Popen(command, env=env, text=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT) as process:
        def relay_output():
            with process.stdout, (workspace / f'{seed}_{name}.log').open('a') as log:
                for line in process.stdout:
                    log.write(line)
                    log.flush()
                    print(line, end='', flush=True)
        relay = Thread(target=relay_output, daemon=True)
        relay.start()
        try:
            status = process.wait(timeout=1980)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            process.kill()
            process.wait()
            raise
        finally:
            relay.join(timeout=5)
    if status:
        raise RuntimeError(f'Training exited with status {status}.')

def report_pair(seed):
    if seed not in SELECTED_SEEDS:
        return
    paths = [workspace / 'results' / f'seed_{seed}_{name}.json'
             for name in ('shallow_wide', 'deep_narrow')]
    records = [json.loads(path.read_text()) for path in paths]
    for key in ('runtime_id', 'threads', 'cpu', 'torch', 'revision', 'train_sha256',
                'eval_sha256', 'target_bytes', 'time_budget_s'):
        if records[0][key] != records[1][key]:
            raise RuntimeError(f'Unmatched {key} for seed {seed}')
    by_model = {record['model']: {row['checkpoint']: row
                                for row in record['measurements']} for record in records}
    print('PAIR RESULT ' + json.dumps({'seed': seed, 'runtime_id': records[0]['runtime_id'],
                                      'models': by_model}), flush=True)


In [ ]:
train_model(1776, 'shallow_wide')

In [ ]:
train_model(1776, 'deep_narrow', predecessor='shallow_wide')

In [ ]:
report_pair(1776)

In [ ]:
train_model(2027, 'deep_narrow')

In [ ]:
train_model(2027, 'shallow_wide', predecessor='deep_narrow')

In [ ]:
report_pair(2027)

In [ ]:
train_model(31415, 'shallow_wide')

In [ ]:
train_model(31415, 'deep_narrow', predecessor='shallow_wide')

In [ ]:
report_pair(31415)